[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week12.ipynb)

# Week 12: Representation Learning
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Autoencoders and latent representations; contrastive and self-supervised methods.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Train an autoencoder and a contrastive embedding.
- Probe and interpret a learned latent space.
- Reason about what makes a representation useful.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Train an autoencoder and visualize reconstructions and the latent space.

In [ ]:
# Train a small autoencoder on MNIST and show reconstructions
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
dl = DataLoader(datasets.MNIST("./data", train=True, download=True, transform=transforms.ToTensor()),
                batch_size=256, shuffle=True)
ae = nn.Sequential(nn.Flatten(), nn.Linear(784, 32), nn.ReLU(),
                   nn.Linear(32, 784), nn.Sigmoid()).to(device)
opt = torch.optim.Adam(ae.parameters(), 1e-3); f = nn.MSELoss()
for epoch in range(2):
    for xb, _ in dl:
        xb = xb.to(device); opt.zero_grad(); f(ae(xb).view_as(xb), xb).backward(); opt.step()
xb, _ = next(iter(dl)); xb = xb.to(device); rec = ae(xb).view_as(xb).detach().cpu()
fig, ax = plt.subplots(2, 5, figsize=(8, 3))
for i in range(5):
    ax[0, i].imshow(xb[i, 0].cpu(), cmap="gray"); ax[0, i].axis("off")
    ax[1, i].imshow(rec[i, 0], cmap="gray"); ax[1, i].axis("off")
plt.show()

## Demonstration 2
Interpolate between two points in latent space live.

In [ ]:
# Interpolate between two examples (the idea behind latent interpolation)
a, b = xb[0].cpu(), xb[1].cpu()
fig, ax = plt.subplots(1, 6, figsize=(9, 2))
for i, alpha in enumerate(torch.linspace(0, 1, 6)):
    ax[i].imshow(((1 - alpha) * a + alpha * b)[0], cmap="gray"); ax[i].axis("off")
plt.show()

## Demonstration 3
Sketch a contrastive setup and show the augmentation views.

In [ ]:
# Two augmented views of one image form a positive pair (contrastive setup)
from torchvision import transforms
aug = transforms.Compose([transforms.RandomResizedCrop(28, scale=(0.6, 1.0)),
                          transforms.RandomHorizontalFlip()])
img = xb[0].cpu()
fig, ax = plt.subplots(1, 3, figsize=(6, 2))
for a_, t, title in zip(ax, [img, aug(img), aug(img)], ["original", "view 1", "view 2"]):
    a_.imshow(t[0], cmap="gray"); a_.set_title(title); a_.axis("off")
plt.show()

---
Student materials for this week: the lab handout (`labs/week12.html`) and the curated references (`references/week12.html`) in the course site.